## Recommendation System

#### 1. Data Preprocessing

In [34]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv(r"C:\Users\suraj\OneDrive\Desktop\data sets\anime.csv")
df

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
...,...,...,...,...,...,...,...
12289,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211
12290,5543,Under World,Hentai,OVA,1,4.28,183
12291,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219
12292,6133,Violence Gekiga Shin David no Hoshi: Inma Dens...,Hentai,OVA,1,4.98,175


In [35]:
# Inspect the data
print(df.head())
print(df.info())

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  

In [36]:
# Drop rows with missing critical information
df = df.dropna(subset=['name', 'genre', 'rating', 'members', 'episodes'])
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


#### 2. Feature Extraction

Convert genres to numerical features

In [37]:
# 1. Split genres into list
df.loc[:, 'genre'] = df['genre'].apply(lambda x: x.split(', ') if isinstance(x, str) else [])

# Show the first 5 rows of genres
print("Genres after splitting into lists:")
print(df['genre'].head(), "\n")

# 2. Convert genres to numeric form
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
genre_features = mlb.fit_transform(df['genre'])

# Show genre columns
print("Genre columns (MultiLabelBinarizer classes):")
print(mlb.classes_, "\n")

# Show first 5 rows of genre_features
print("First 5 rows of genre_features (binary):")
print(genre_features[:5], "\n")

# 3. Combine with numeric features
import numpy as np
numeric_features = df[['rating', 'members', 'episodes']].fillna(0).values
features = np.hstack([genre_features, numeric_features])

# Show combined features
print("Shape of combined features matrix:", features.shape)
print("First 5 rows of features matrix:")
print(features[:5])

Genres after splitting into lists:
0               [Drama, Romance, School, Supernatural]
1    [Action, Adventure, Drama, Fantasy, Magic, Mil...
2    [Action, Comedy, Historical, Parody, Samurai, ...
3                                   [Sci-Fi, Thriller]
4    [Action, Comedy, Historical, Parody, Samurai, ...
Name: genre, dtype: object 

Genre columns (MultiLabelBinarizer classes):
['Action' 'Adventure' 'Cars' 'Comedy' 'Dementia' 'Demons' 'Drama' 'Ecchi'
 'Fantasy' 'Game' 'Harem' 'Hentai' 'Historical' 'Horror' 'Josei' 'Kids'
 'Magic' 'Martial Arts' 'Mecha' 'Military' 'Music' 'Mystery' 'Parody'
 'Police' 'Psychological' 'Romance' 'Samurai' 'School' 'Sci-Fi' 'Seinen'
 'Shoujo' 'Shoujo Ai' 'Shounen' 'Shounen Ai' 'Slice of Life' 'Space'
 'Sports' 'Super Power' 'Supernatural' 'Thriller' 'Vampire' 'Yaoi' 'Yuri'] 

First 5 rows of genre_features (binary):
[[0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0
  0 0 1 0 0 0 0]
 [1 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0

#### 3. Recommendation System Using Cosine Similarity

In [38]:

df = df[['name','genre','rating','members','episodes']].dropna()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(0)
df['members'] = pd.to_numeric(df['members'], errors='coerce').fillna(0)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce').fillna(0)
df['genre'] = df['genre'].apply(lambda x: x.split(', ') if isinstance(x,str) else [])

# Encode genres
mlb = MultiLabelBinarizer()
genre_features = mlb.fit_transform(df['genre'])

# Combine features
numeric_features = StandardScaler().fit_transform(df[['rating','members','episodes']])
features = np.hstack([genre_features, numeric_features]).astype(float)

# Compute cosine similarity
cosine_sim = cosine_similarity(features)

# Recommendation function
def recommend_anime(anime_title, df=df, sim=cosine_sim, top_n=10):
    if anime_title not in df['name'].values: return "Anime not found"
    idx = df.index[df['name']==anime_title][0]
    sim_scores = [(i,s) for i,s in enumerate(sim[idx]) if i!=idx]
    sim_scores = sorted(sim_scores, key=lambda x:x[1], reverse=True)[:top_n]
    return df['name'].iloc[[i for i,_ in sim_scores]].tolist()

# Example usage
recommend_anime("Naruto")



['Fairy Tail',
 'Dragon Ball GT',
 'Gyakuten Saiban: Sono &quot;Shinjitsu&quot;, Igi Ari!',
 'Trickster: Edogawa Ranpo &quot;Shounen Tanteidan&quot; yori',
 'D.Gray-man',
 'Sousei no Onmyouji',
 'Captain Earth',
 'Hunter x Hunter (2011)',
 'Dragon Ball',
 'Seikon no Qwaser']


4. Evaluation (Optional for Content-Based

For content-based systems:

Split into train/test based on anime popularity or user ratings.
Metrics like precision and recall can be evaluated by comparing recommendations to user-rated “liked” anime.
For collaborative filtering, metrics are more straightforward.

In [39]:
# Example splitting (if user data exists)
train, test = train_test_split(df, test_size=0.2, random_state=42)

#### 5. Interview Questions

#### Q1. Difference between user-based and item-based collaborative filtering:

User-based CF: Recommends items liked by similar users.
Item-based CF: Recommends items similar to items the user has liked.
Key difference: User-based finds similar users, item-based finds similar items.
#### Q2. What is collaborative filtering, and how does it work?

Collaborative filtering is a recommendation method that predicts a user’s interests based on past behavior of many users.
Works by finding patterns in user-item interactions. Example: if users A & B like the same anime, anime liked by A may be recommended to B.